In [1]:
from __future__ import annotations

import uuid
from enum import Enum
import errno
import sys
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

from debugpy.common.log import LEVELS

In [2]:
from identifier import Id, CompositeId
from meta import Run, Dir, File
from db import Db

In [ ]:
db_name = 'test-db-for-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

the_db = Db.open(path, readonly=readonly, create=create)
env = the_db.env

In [4]:
# import identifier
# reload(identifier)
# from identifier import Id, CompositeId
#
# import db
# reload(db)
# from db import Db

In [5]:
# show lmdb env info

# env.flags(), env.path(), env.info()

In [6]:
# env.close()

In [18]:
env.stat()['entries'], the_db.max_run_id()

(11, 42)

In [17]:
# list all key-value pairs in the db, using lmdb api

with env.begin(write=False) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            print(f'key={key}', f'value={value}')

key=b'd:\x00*\x00\x01' value=b'\x08\x01\x12\x04\x00*\x00\x01"\rthis\\is\\a\\dir-f\xe6@F2\x101111222233334444:\x105555666677778888'
key=b'd:\x00*\x00\x02' value=b'\x08\x01\x12\x04\x00*\x00\x02\x1a\x04\x00*\x00\x01"\x11this\\is\\a\\dir\\too-f\xe6@F2\x101111222233334444:\x109999aaaabbbbcccc'
key=b'dhd:UUffww\x88\x88:\x11\x11""33DD:\x00*\x00\x01' value=b''
key=b'dhd:\x99\x99\xaa\xaa\xbb\xbb\xcc\xcc:\x11\x11""33DD:\x00*\x00\x02' value=b''
key=b'dhf:\x11\x11""33DD:UUffww\x88\x88:\x00*\x00\x01' value=b''
key=b'dhf:\x11\x11""33DD:\x99\x99\xaa\xaa\xbb\xbb\xcc\xcc:\x00*\x00\x02' value=b''
key=b'f:\x00*\x00\x01\x00\x01' value=b'\x08\x01\x12\x06\x00*\x00\x01\x00\x01\x1a\tsome_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'f:\x00*\x00\x02\x00\x01' value=b'\x08\x01\x12\x06\x00*\x00\x02\x00\x01\x1a\x0canother_file%\xcd\xc0\xa9E(\xb2\xf2\x192 abcd1234abcd1234abcd1234abcd1234'
key=b'fh:\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124\xab\xcd\x124:\x00*\x00\x01\x00\x01' value=b''
key=b'

In [9]:
# create a Run obj

run42 = Run(
    id=Id(42),
    uuid=uuid.uuid4(),
    path=Path('C:\\some\\start\\dir'),
    description='run from jupyter for development',
    platform='lekker plat',
    start_time=1.2,
    end_time=2.4,
    duration=1.2,
    status='initialized',
    root_dir=None,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    tags=['ma', 'riet']
)

In [10]:
# create Dir obj's

dir1 = Dir(
    run = run42,
    id = CompositeId(run42.id),
    path=Path('this/is/a/dir'),
    path_repr='reppur',
    timestamp=12345.6,
    files_hash='1111222233334444',
    dirs_hash='5555666677778888',
    all_hash='9999aaaabbbbcccc',
)

run42.root_dir = dir1

dir2 = Dir(
    run = run42,
    id = CompositeId(run42.id),
    path=Path('this/is/a/dir/too'),
    path_repr='reppur2',
    parent=dir1,
    timestamp=12345.6,
    files_hash='1111222233334444',
    dirs_hash='9999aaaabbbbcccc',
    all_hash='5555666677778888',
)

In [11]:
# create File obj's

file1 = File(
    run = run42,
    id = CompositeId(dir1.id),
    name = 'some_file',
    dir=dir1,
    creation_time=5432.1,
    length=424242,
    hash='abcd1234' * 4,
)

file2 = File(
    run = run42,
    id = CompositeId(dir2.id),
    name = 'another_file',
    dir=dir2,
    creation_time=5432.1,
    length=424242,
    hash='abcd1234' * 4,
)

In [12]:
# Store Run obj

the_db.put_run(run42)

Stored run 42


In [13]:
# Restore Run obj from db

# run_id = Id(42)
# rrun42 = the_db.get_run(run_id)
# rrun42.id

In [14]:
# store Dir obj's

the_db.put_dir(dir1)
the_db.put_dir(dir2)

Stored dir (42, 1)
Stored dir (42, 2)


In [15]:
# store File obj's

the_db.put_file(file1)
the_db.put_file(file2)

Stored file (42, 1, 1)
Stored file (42, 2, 1)
